# Entrenamiento del modelo MOFA2
**TFM — *Prunus dulcis* cv. Nonpareil | Dormancia**

Entrena un modelo MOFA2 sobre **5 vistas** (Argelaguet et al. 2018, 2020):

El modelo entrenado se guarda como `mofa2_model.hdf5`, compatible con la librería R de MOFA2:
```r
library(MOFA2)
model <- load_model("mofa2_model.hdf5")
```

## 0. Parámetros

In [ ]:
RNASEQ_FILE   = "Expression_Profile_Nonpareil_gene.xlsx"
MIRNA_FILE    = "Expression_Profile_ppe_Prunus_dulcis_Lauranne_miRNA.xlsx"
METHYL_CG     = "methyl_CG_promoter.csv"
METHYL_CHG    = "methyl_CHG_promoter.csv"
METHYL_CHH    = "methyl_CHH_promoter.csv"
METADATA_FILE = "metadatos.xlsx"
OUTPUT_HDF5   = "mofa2_model.hdf5"

# Parámetros del modelo (mismos que produjeron el modelo del TFM)
N_FACTORS  = 10      # factores latentes
TOP_RNASEQ = 5000    # top genes por varianza tras VST
SEED       = 42

# Muestras excluidas por QC
EXCLUDE_RNASEQ = [10]      # outlier: 5.9M reads, tasa mapeo 59%
EXCLUDE_WGBS   = [3, 38]   # baja tasa de conversión bisulfito

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from mofapy2.run.entry_point import entry_point

print("Imports OK")

## 2. Metadatos

In [ ]:
meta = pd.read_excel(METADATA_FILE)
meta = meta.dropna(subset=['ID'])
meta['ID'] = meta['ID'].astype(int)
meta['Dormancy state'] = meta['Dormancy state'].str.strip()
meta = meta.drop_duplicates(subset='ID')
meta = meta[meta['ID'] <= 39].copy()

treatment_map = {None: 'Control', 'Control': 'Control',
                 'Asc': 'Asc', 'Eth': 'Eth', 'Ethephon': 'Eth'}
meta['group'] = meta['Treatment'].fillna('Control').map(
    lambda x: treatment_map.get(str(x).strip(), 'Control'))

print(f"{len(meta)} muestras cargadas")
print(meta.groupby('group')['Dormancy state'].value_counts())

## 3. Vista RNA-seq — VST, top 5.000 genes por varianza

In [ ]:
expr = pd.read_excel(RNASEQ_FILE)
count_cols = [c for c in expr.columns if c.endswith('_Read_Count')]
counts_rna = expr[['Gene_ID'] + count_cols].copy()
counts_rna.columns = ['Gene_ID'] + [int(c.replace('_Read_Count', ''))
                                     for c in count_cols]
counts_rna = counts_rna.set_index('Gene_ID').T  # muestras x genes

# Excluir S10 (outlier)
counts_rna = counts_rna.loc[
    [s for s in counts_rna.index if s not in EXCLUDE_RNASEQ]
].astype(int)

# Filtro expresión mínima
keep = (counts_rna >= 10).sum(axis=0) >= 3
counts_rna = counts_rna.loc[:, keep]
print(f"Genes tras filtro: {counts_rna.shape[1]}")

In [ ]:
# Metadatos para pyDESeq2
counts_rna.index = counts_rna.index.astype(str)
meta_rna = meta[meta['ID'].isin(counts_rna.index.astype(int))].copy()
meta_rna = meta_rna.set_index(meta_rna['ID'].astype(str)).loc[counts_rna.index]
meta_rna['dormancy_state'] = pd.Categorical(
    meta_rna['Dormancy state'],
    categories=['Endodormancy', 'Endodormancy release', 'Ecodormancy'])
meta_rna['treatment'] = pd.Categorical(
    meta_rna['Treatment'].fillna('Control').astype(str),
    categories=['Control', 'Asc', 'Eth'])

# VST con pyDESeq2
inference = DefaultInference(n_cpus=4)
dds_rna = DeseqDataSet(
    counts=counts_rna,
    metadata=meta_rna[['dormancy_state', 'treatment']],
    design_factors=['dormancy_state', 'treatment'],
    inference=inference,
    quiet=True
)
dds_rna.deseq2()
vst_rna = np.log2(dds_rna.layers['normed_counts'] + 1)
vst_rna_df = pd.DataFrame(vst_rna, index=counts_rna.index, columns=counts_rna.columns)

# Top 5.000 por varianza
top_genes = vst_rna_df.var(axis=0).nlargest(TOP_RNASEQ).index
vst_rna_df = vst_rna_df[top_genes]
print(f"Matriz RNA-seq final: {vst_rna_df.shape}")

## 4. Vista miRNA — VST, 77 miRNAs detectables

In [ ]:
mirna_raw = pd.read_excel(MIRNA_FILE)
count_cols_m = [c for c in mirna_raw.columns if c.endswith('_Read_Count')]
counts_mirna = mirna_raw[['Mature_ID'] + count_cols_m].copy()
counts_mirna.columns = ['miRNA'] + [int(c.replace('_Read_Count', ''))
                                     for c in count_cols_m]
counts_mirna = counts_mirna.set_index('miRNA').T.astype(int)

# Filtro >= 10 reads en >= 3 muestras → 77 miRNAs
keep_m = (counts_mirna >= 10).sum(axis=0) >= 3
counts_mirna = counts_mirna.loc[:, keep_m]
print(f"miRNAs detectables: {counts_mirna.shape[1]}")

In [ ]:
counts_mirna.index = counts_mirna.index.astype(str)
meta_mirna = meta.set_index(meta['ID'].astype(str)).loc[counts_mirna.index].copy()
meta_mirna['dormancy_state'] = pd.Categorical(
    meta_mirna['Dormancy state'],
    categories=['Endodormancy', 'Endodormancy release', 'Ecodormancy'])
meta_mirna['treatment'] = pd.Categorical(
    meta_mirna['Treatment'].fillna('Control').astype(str),
    categories=['Control', 'Asc', 'Eth'])

dds_mirna = DeseqDataSet(
    counts=counts_mirna,
    metadata=meta_mirna[['dormancy_state', 'treatment']],
    design_factors=['dormancy_state', 'treatment'],
    inference=inference,
    quiet=True
)
dds_mirna.deseq2()
vst_mirna = np.log2(dds_mirna.layers['normed_counts'] + 1)
vst_mirna_df = pd.DataFrame(vst_mirna, index=counts_mirna.index, columns=counts_mirna.columns)
print(f"Matriz miRNA final: {vst_mirna_df.shape}")

## 5. Vistas metilación — CG, CHG, CHH promotor

In [ ]:
methyl = {}
for ctx, filepath in [('CG_promoter', METHYL_CG),
                       ('CHG_promoter', METHYL_CHG),
                       ('CHH_promoter', METHYL_CHH)]:
    df = pd.read_csv(filepath, index_col=0)
    df.index = df.index.astype(str)
    df = df[~df.index.isin([str(s) for s in EXCLUDE_WGBS])]
    methyl[ctx] = df
    print(f"{ctx}: {df.shape[0]} muestras x {df.shape[1]:,} genes")

## 6. Construir diccionario de datos para MOFA2

In [ ]:
sample_to_group = dict(zip(meta['ID'].astype(str), meta['group']))
groups = ['Control', 'Asc', 'Eth']

views_dict = {
    'RNA-seq':      vst_rna_df,
    'miRNA':        vst_mirna_df,
    'CG_promoter':  methyl['CG_promoter'],
    'CHG_promoter': methyl['CHG_promoter'],
    'CHH_promoter': methyl['CHH_promoter'],
}

# Formato MOFA2: {vista: {grupo: matrix features x samples}}
data_mofa = {}
for view_name, mat in views_dict.items():
    data_mofa[view_name] = {}
    for grp in groups:
        grp_samples = [s for s in mat.index if sample_to_group.get(s) == grp]
        data_mofa[view_name][grp] = mat.loc[grp_samples].T if grp_samples else None

# Resumen de dimensiones por vista y grupo
print(f"{'Vista':<15} {'Control':>10} {'Asc':>8} {'Eth':>8}")
print("-" * 45)
for v in groups:
    pass
for view in views_dict:
    dims = []
    for grp in groups:
        m = data_mofa[view][grp]
        dims.append(f"{m.shape[1]}s" if m is not None else "NA")
    print(f"{view:<15} {dims[0]:>10} {dims[1]:>8} {dims[2]:>8}")

## 7. Entrenar modelo MOFA2

**Esta celda puede tardar 5–15 minutos.** El modelo del TFM convergió en 181 iteraciones.

In [ ]:
ent = entry_point()

# Opciones de datos
ent.set_data_options(scale_views=False, scale_groups=False)
ent.set_data_matrix(
    data_mofa,
    likelihoods=['gaussian'] * 5,
    views_names=list(views_dict.keys()),
    groups_names=groups
)

# Opciones del modelo — K=10, spike-and-slab, ARD activado
ent.set_model_options(
    factors=N_FACTORS,
    spikeslab_weights=True,
    ard_factors=True,
    ard_weights=True
)

# Opciones de entrenamiento — convergencia medium (tol ELBO relativa < 0.0005)
ent.set_train_options(
    iter=2000,
    convergence_mode='medium',
    startELBO=1,
    freqELBO=1,
    dropR2=0.001,
    seed=SEED,
    verbose=True
)

print(f"Parámetros: K={N_FACTORS} factores | seed={SEED} | convergencia=medium")
print("Entrenando...\n")
ent.build()
ent.run()

## 8. Guardar modelo

In [ ]:
ent.save(OUTPUT_HDF5)
print(f"Modelo guardado: {OUTPUT_HDF5}")
print()
print("Para cargar en R:")
print('  library(MOFA2)')
print(f'  model <- load_model("{OUTPUT_HDF5}")')